# ILC level 2 centroids + smart features KMeans

Этот ноутбук кластеризует статьи по `smart_features`, используя узлы ILC второго уровня как фиксированные центроиды для KMeans.

Логика:

1. загрузить статьи с NER/keywords;
2. собрать `smart_features = keywords + meaningful NER`;
3. взять узлы ILC на глубине 2;
4. построить embeddings статей и ILC-центроидов одной моделью;
5. запустить KMeans с `init=ilc_level2_centroids`, `n_init=1`, `max_iter=1`;
6. сохранить статьи с `ilc_level2_cluster`, `ilc_notation`, `ilc_label`, `min_distance_to_ilc_center`.

In [9]:
USE_DIM_REDUCTION = True
DIM_REDUCTION_DIM = 10

In [10]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
import re
from collections import Counter
import matplotlib.pyplot as plt
from wordcloud import WordCloud

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, pairwise_distances, silhouette_score
from sklearn.preprocessing import StandardScaler


In [22]:
# Configuration paths and constants
DATA_DIR = Path("../data")
ARTIFACTS = DATA_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

ARTICLES_CSV = ARTIFACTS / "articles_with_ilc_classes.csv"
SMART_ARTICLES_CSV = ARTIFACTS / "articles_with_smart_features.csv"
WITH_ORCID_CSV = Path("../misc/orcid_profiles_with_ner.csv")
SMART_WITH_ORCID_CSV = ARTIFACTS / "smart_with_orcid.csv"
OUTPUT_CSV = ARTIFACTS / "articles_with_ilc_level2_kmeans.csv"
WITH_ORCID_LEVEL2_CSV = ARTIFACTS / "with_orcid_level2_assignments.csv"

ILC_JSON = Path("../parse/ilc_tree_depth3.json")
ILC_LEVEL2_METADATA_PATH = ARTIFACTS / "ilc_level2_centroids.pkl"

CLUSTERING_QUALITY_CSV = ARTIFACTS / "ilc_level2_clustering_quality.csv"
PCA_CLUSTERING_QUALITY_CSV = ARTIFACTS / "pca_clustering_quality.csv"
AUTHOR_TOPK_CSV = ARTIFACTS / "author_topk_ilc_centroids.csv"
AUTHOR_WORKS_SUMMARY_CSV = ARTIFACTS / "author_works_summary.csv"

ARTICLE_EMBEDDINGS_PATH = ARTIFACTS / "article_embeddings.npy"
ILC_LEVEL2_CENTROIDS_PATH = ARTIFACTS / "ilc_level2_centroids.npy"
AUTHOR_EMBEDDINGS_PATH = ARTIFACTS / "author_embeddings.npy"
WITH_ORCID_EMBEDDINGS_PATH = ARTIFACTS / "with_orcid_embeddings.npy"

MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
BATCH_SIZE = 32
TOP_K_CENTROIDS = 5
QUALITY_SAMPLE_SIZE = 10000

print("Configuration initialized.")
print(f"Data directory: {DATA_DIR}")
print(f"Artifacts directory: {ARTIFACTS}")

Configuration initialized.
Data directory: ..\data
Artifacts directory: ..\data\artifacts


## Загрузка статей и сбор smart features

In [12]:
MEANINGLESS_NER_CATEGORIES = {"DATE", "TIME", "PERCENT", "MONEY", "QUANTITY", "CARDINAL", "ORDINAL"}

def parse_keywords(value):
    if pd.isna(value) or not isinstance(value, str):
        return []
    words = []
    for item in value.split("; "):
        try:
            keyword, _ = item.split("|")
            keyword = keyword.strip()
            if keyword:
                words.append(keyword)
        except ValueError:
            continue
    return words

def parse_ner(value):
    if pd.isna(value) or not isinstance(value, str):
        return []
    words = []
    for item in value.split("; "):
        try:
            word, label, _ = item.split("|")
            word = word.strip()
            if word and label not in MEANINGLESS_NER_CATEGORIES:
                words.append(word)
        except ValueError:
            continue
    return words

def combine_smart_features(row):
    return " ".join(parse_keywords(row.get("keywords")) + parse_ner(row.get("ner_entities")))

df_articles = pd.read_csv(ARTICLES_CSV)
print("Статей:", len(df_articles))

if "smart_features" not in df_articles.columns:
    tqdm.pandas(desc="smart_features")
    df_articles["smart_features"] = df_articles.progress_apply(combine_smart_features, axis=1)

df_articles.to_csv(SMART_ARTICLES_CSV, index=False)
print("Сохранено:", SMART_ARTICLES_CSV)
df_articles[["oai_identifier", "smart_features"]].head()

Статей: 124282


smart_features: 100%|██████████| 124282/124282 [00:01<00:00, 89766.01it/s]


Сохранено: ..\data\artifacts\articles_with_smart_features.csv


,oai_identifier,smart_features
0,oai:dspace.mit.edu:1721.1/126771,computing 1980s finance visicalc pioneering ma...
1,oai:dspace.mit.edu:1721.1/161710,2008 dean student report president year studen...
2,oai:dspace.mit.edu:1721.1/11248,anisotropy dimensional printing structural cer...
3,oai:dspace.mit.edu:1721.1/3029,parallel processing communication aspects supp...
4,oai:dspace.mit.edu:1721.1/8302,dna electrophoresis microdevices filled polyme...


## ILC level 2: сбор центроидов

In [13]:
with open(ILC_JSON, "r", encoding="utf-8") as f:
    ilc_tree = json.load(f)

def collect_nodes_at_depth(node, current_depth=1, target_depth=2):
    if current_depth == target_depth:
        return [node]
    if current_depth > target_depth:
        return []
    nodes = []
    for child in node.get("children", []):
        nodes.extend(collect_nodes_at_depth(child, current_depth + 1, target_depth))
    return nodes

level2_nodes = []
for root in ilc_tree:
    level2_nodes.extend(collect_nodes_at_depth(root, current_depth=1, target_depth=2))

def build_ilc_text(node):
    parts = [node.get("description", ""), node.get("label", "")]
    return " ".join(str(part).strip() for part in parts if str(part).strip())

df_centroids = pd.DataFrame([
    {
        "notation": node.get("notation"),
        "label": node.get("label"),
        "combined_text": build_ilc_text(node),
    }
    for node in level2_nodes
])

df_centroids.to_pickle(ILC_LEVEL2_METADATA_PATH)
print("ILC level 2 центроидов:", len(df_centroids))
print("Сохранено:", ILC_LEVEL2_METADATA_PATH)
df_centroids.head()

ILC level 2 центроидов: 333
Сохранено: ..\data\artifacts\ilc_level2_centroids.pkl


,notation,label,combined_text
0,UX,persons by birth millennium,persons by birth millennium
1,XA,x0,x0
2,XB,x1,x1
3,XC,x2,x2
4,XX,what,what


## Векторизация smart features и ILC level 2 центроидов

In [14]:
def encode_texts(texts, model, batch_size=32, desc="embeddings"):
    embeddings = []
    for start in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[start:start + batch_size]
        embeddings.extend(model.encode(batch, show_progress_bar=False))
    return np.asarray(embeddings, dtype="float32")

model = SentenceTransformer(MODEL_NAME)

if ARTICLE_EMBEDDINGS_PATH.exists():
    article_embeddings = np.load(ARTICLE_EMBEDDINGS_PATH)
    if article_embeddings.shape[0] != len(df_articles):
        print("Размер старых embeddings не совпадает с числом статей, пересчитываю...")
        article_embeddings = encode_texts(df_articles["smart_features"].fillna("").astype(str).tolist(), model, BATCH_SIZE, "article embeddings")
        np.save(ARTICLE_EMBEDDINGS_PATH, article_embeddings)
else:
    article_embeddings = encode_texts(df_articles["smart_features"].fillna("").astype(str).tolist(), model, BATCH_SIZE, "article embeddings")
    np.save(ARTICLE_EMBEDDINGS_PATH, article_embeddings)

if ILC_LEVEL2_CENTROIDS_PATH.exists():
    ilc_centroids = np.load(ILC_LEVEL2_CENTROIDS_PATH)
    if ilc_centroids.shape[0] != len(df_centroids):
        print("Размер старых ILC embeddings не совпадает с числом центроидов, пересчитываю...")
        ilc_centroids = encode_texts(df_centroids["combined_text"].fillna("").astype(str).tolist(), model, BATCH_SIZE, "ILC level 2 centroids")
        np.save(ILC_LEVEL2_CENTROIDS_PATH, ilc_centroids)
else:
    ilc_centroids = encode_texts(df_centroids["combined_text"].fillna("").astype(str).tolist(), model, BATCH_SIZE, "ILC level 2 centroids")
    np.save(ILC_LEVEL2_CENTROIDS_PATH, ilc_centroids)

print("article_embeddings:", article_embeddings.shape)
print("ilc_centroids:", ilc_centroids.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4774.16it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
article embeddings: 100%|██████████| 3884/3884 [27:37<00:00,  2.34it/s]


Размер старых ILC embeddings не совпадает с числом центроидов, пересчитываю...


ILC level 2 centroids: 100%|██████████| 11/11 [00:00<00:00, 19.68it/s]

article_embeddings: (124282, 768)
ilc_centroids: (333, 768)


## Dimensionality reduction with PCA

Apply PCA to the normalized article embeddings and ILC centroids before clustering. This reduces the embedding dimensionality and allows evaluating how much structure is preserved.

In [16]:
if USE_DIM_REDUCTION:
    scaler_pca = StandardScaler()
    article_embeddings_scaled_for_pca = scaler_pca.fit_transform(article_embeddings)
    ilc_centroids_scaled_for_pca = scaler_pca.transform(ilc_centroids)

    pca = PCA(n_components=min(DIM_REDUCTION_DIM, article_embeddings_scaled_for_pca.shape[1]), random_state=42)
    article_embeddings_reduced = pca.fit_transform(article_embeddings_scaled_for_pca)
    ilc_centroids_reduced = pca.transform(ilc_centroids_scaled_for_pca)
    print(f"PCA components: {article_embeddings_reduced.shape[1]}")
    print(f"Explained variance ratio sum: {pca.explained_variance_ratio_.sum():.4f}")
else:
    article_embeddings_reduced = article_embeddings
    ilc_centroids_reduced = ilc_centroids
    print(f"Dimensionality reduction disabled, using original embedding dimension: {article_embeddings_reduced.shape[1]}")

np.save(ARTIFACTS / "article_embeddings_pca.npy", article_embeddings_reduced)
np.save(ARTIFACTS / "ilc_level2_centroids_pca.npy", ilc_centroids_reduced)

kmeans_pca = KMeans(
    n_clusters=len(ilc_centroids_reduced),
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=42,
)

pca_labels = kmeans_pca.fit_predict(article_embeddings_reduced)
pca_distances = pairwise_distances(article_embeddings_reduced, kmeans_pca.cluster_centers_, metric="cosine")

# Calculate nearest original ILC centroid for comparison
pca_distances_to_original = pairwise_distances(article_embeddings_reduced, ilc_centroids_reduced, metric="cosine")
nearest_original_centroid = pca_distances_to_original.argmin(axis=1)

pca_quality = {
    "n_articles": int(len(pca_labels)),
    "n_centroids": int(len(kmeans_pca.cluster_centers_)),
    "n_used_clusters": int(len(np.unique(pca_labels))),
    "n_empty_clusters": int(len(kmeans_pca.cluster_centers_) - len(np.unique(pca_labels))),
    "mean_assigned_cosine_distance": float(np.mean(pca_distances.min(axis=1))),
    "median_assigned_cosine_distance": float(np.median(pca_distances.min(axis=1))),
    "p95_assigned_cosine_distance": float(np.percentile(pca_distances.min(axis=1), 95)),
    "kmeans_matches_nearest_original_centroid_ratio": float(np.mean(pca_labels == nearest_original_centroid)),
}

metric_embeddings_pca = article_embeddings_reduced
metric_labels_pca = pca_labels
if len(metric_labels_pca) > QUALITY_SAMPLE_SIZE:
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(metric_labels_pca), size=QUALITY_SAMPLE_SIZE, replace=False)
    metric_embeddings_pca = metric_embeddings_pca[sample_idx]
    metric_labels_pca = metric_labels_pca[sample_idx]

pca_quality["quality_sample_size"] = int(len(metric_labels_pca))
metric_unique_labels_pca = np.unique(metric_labels_pca)
if 1 < len(metric_unique_labels_pca) < len(metric_labels_pca):
    pca_quality["silhouette_cosine"] = float(silhouette_score(metric_embeddings_pca, metric_labels_pca, metric="cosine"))
    pca_quality["davies_bouldin"] = float(davies_bouldin_score(metric_embeddings_pca, metric_labels_pca))
    pca_quality["calinski_harabasz"] = float(calinski_harabasz_score(metric_embeddings_pca, metric_labels_pca))
else:
    pca_quality["silhouette_cosine"] = np.nan
    pca_quality["davies_bouldin"] = np.nan
    pca_quality["calinski_harabasz"] = np.nan

pca_quality_df = pd.DataFrame([pca_quality])

pca_quality_df.to_csv(PCA_CLUSTERING_QUALITY_CSV, index=False)

print("PCA KMeans quality saved:", PCA_CLUSTERING_QUALITY_CSV)
pca_quality_df

PCA components: 10
Explained variance ratio sum: 0.2670
PCA KMeans quality saved: ..\data\artifacts\pca_clustering_quality.csv


,n_articles,n_centroids,n_used_clusters,n_empty_clusters,mean_assigned_cosine_distance,median_assigned_cosine_distance,p95_assigned_cosine_distance,kmeans_matches_nearest_original_centroid_ratio,quality_sample_size,silhouette_cosine,davies_bouldin,calinski_harabasz
0,124282,333,333,0,0.0839,0.072596,0.187199,0.001786,10000,0.11883,1.759396,159.069019


## KMeans с ILC level 2 центроидами

In [17]:
assert article_embeddings_reduced.shape[0] == len(df_articles), "Число embeddings статей не совпадает с числом строк CSV"
assert article_embeddings_reduced.shape[1] == ilc_centroids_reduced.shape[1], "Размерность article и ILC embeddings различается"

kmeans = KMeans(
    n_clusters=len(ilc_centroids_reduced),
    init=ilc_centroids_reduced,
    n_init=1,
    max_iter=1,
    random_state=42,
)

labels = kmeans.fit_predict(article_embeddings_reduced)
distances = pairwise_distances(article_embeddings_reduced, ilc_centroids_reduced, metric="cosine")

df_result = df_articles.copy()
df_result["ilc_level2_cluster"] = labels
df_result["min_distance_to_ilc_center"] = distances.min(axis=1)
df_result["ilc_notation"] = df_result["ilc_level2_cluster"].map(lambda i: df_centroids.iloc[int(i)]["notation"])
df_result["ilc_label"] = df_result["ilc_level2_cluster"].map(lambda i: df_centroids.iloc[int(i)]["label"])

df_result.to_csv(OUTPUT_CSV, index=False)
print("Сохранено:", OUTPUT_CSV)
df_result[["oai_identifier", "ilc_level2_cluster", "ilc_notation", "ilc_label", "min_distance_to_ilc_center"]].head()

Сохранено: ..\data\artifacts\articles_with_ilc_level2_kmeans.csv


,oai_identifier,ilc_level2_cluster,ilc_notation,ilc_label,min_distance_to_ilc_center
0,oai:dspace.mit.edu:1721.1/126771,15,ag,operations,0.056363
1,oai:dspace.mit.edu:1721.1/161710,5,ZA,the aforementioned,0.079504
2,oai:dspace.mit.edu:1721.1/11248,94,gs,solids,0.204206
3,oai:dspace.mit.edu:1721.1/3029,24,at,algorithms,0.267054
4,oai:dspace.mit.edu:1721.1/8302,35,cd,membranes,0.420779


## Быстрая сводка по кластерам

In [18]:
summary = (
    df_result.groupby(["ilc_notation", "ilc_label"])
    .size()
    .reset_index(name="articles_count")
    .sort_values("articles_count", ascending=False)
)

SUMMARY_CSV = ARTIFACTS / "articles_with_ilc_level2_summary.csv"
summary.to_csv(SUMMARY_CSV, index=False)
print("Сводка сохранена:", SUMMARY_CSV)
summary.head(20)

Сводка сохранена: ..\data\artifacts\articles_with_ilc_level2_summary.csv


,ilc_notation,ilc_label,articles_count
77,fp,lipids,2415
81,fx,solid state compounds,2288
280,vt,manufacturing,2277
299,ww,settlements,2268
208,qw,language transmission systems,2222
24,at,algorithms,2122
261,uo,corporations,2039
296,ws,buildings,1996
94,gs,solids,1911
87,ggWl,fluids,1895


## Clustering quality for current ILC level 2 centroids

Metrics are computed for article embeddings assigned to ILC level 2 centroids. The block also checks how often KMeans labels match the nearest original ILC centroid by cosine distance.

In [20]:
unique_labels = np.unique(labels)
assigned_distances = distances[np.arange(len(labels)), labels]
nearest_original_centroid = distances.argmin(axis=1)

quality = {
    "n_articles": int(len(labels)),
    "n_centroids": int(len(ilc_centroids_reduced)),
    "n_used_clusters": int(len(unique_labels)),
    "n_empty_clusters": int(len(ilc_centroids_reduced) - len(unique_labels)),
    "mean_assigned_cosine_distance": float(np.mean(assigned_distances)),
    "median_assigned_cosine_distance": float(np.median(assigned_distances)),
    "p95_assigned_cosine_distance": float(np.percentile(assigned_distances, 95)),
    "kmeans_matches_nearest_original_centroid_ratio": float(np.mean(labels == nearest_original_centroid)),
}

metric_embeddings = article_embeddings_reduced
metric_labels = labels
if len(metric_labels) > QUALITY_SAMPLE_SIZE:
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(metric_labels), size=QUALITY_SAMPLE_SIZE, replace=False)
    metric_embeddings = metric_embeddings[sample_idx]
    metric_labels = metric_labels[sample_idx]

quality["quality_sample_size"] = int(len(metric_labels))
metric_unique_labels = np.unique(metric_labels)
if 1 < len(metric_unique_labels) < len(metric_labels):
    quality["silhouette_cosine"] = float(silhouette_score(metric_embeddings, metric_labels, metric="cosine"))
    quality["davies_bouldin"] = float(davies_bouldin_score(metric_embeddings, metric_labels))
    quality["calinski_harabasz"] = float(calinski_harabasz_score(metric_embeddings, metric_labels))
else:
    quality["silhouette_cosine"] = np.nan
    quality["davies_bouldin"] = np.nan
    quality["calinski_harabasz"] = np.nan

quality_df = pd.DataFrame([quality])
quality_df.to_csv(CLUSTERING_QUALITY_CSV, index=False)
print("Quality saved:", CLUSTERING_QUALITY_CSV)
quality_df

Quality saved: ..\data\artifacts\ilc_level2_clustering_quality.csv


,n_articles,n_centroids,n_used_clusters,n_empty_clusters,mean_assigned_cosine_distance,median_assigned_cosine_distance,p95_assigned_cosine_distance,kmeans_matches_nearest_original_centroid_ratio,quality_sample_size,silhouette_cosine,davies_bouldin,calinski_harabasz
0,124282,333,333,0,0.272159,0.230579,0.612932,0.521789,10000,0.052949,1.836095,115.364325


## Authors with ORCID: load articles and build smart features

This block selects only records with `orcid`, then uses the same `smart_features` logic: keywords plus meaningful NER entities.

In [24]:
df_with_orcid = pd.read_csv(WITH_ORCID_CSV)
df_with_orcid = df_with_orcid.dropna(subset=["orcid"]).copy()
df_with_orcid["orcid"] = df_with_orcid["orcid"].astype(str)

if "smart_features" not in df_with_orcid.columns:
    tqdm.pandas(desc="author article smart_features")
    df_with_orcid["smart_features"] = df_with_orcid.progress_apply(combine_smart_features, axis=1)

df_with_orcid.to_csv(SMART_WITH_ORCID_CSV, index=False)
print("Articles with ORCID:", len(df_with_orcid))
print("Unique ORCID:", df_with_orcid["orcid"].nunique())
print("Saved:", SMART_WITH_ORCID_CSV)
print("Available columns:", df_with_orcid.columns.tolist())
df_with_orcid.head()

author article smart_features: 100%|██████████| 5770/5770 [00:00<00:00, 40061.20it/s]


Articles with ORCID: 5770
Unique ORCID: 5770
Saved: ..\data\artifacts\smart_with_orcid.csv
Available columns: ['orcid', 'title', 'description', 'entities', 'smart_features']


,orcid,title,description,entities,smart_features
0,http://orcid.org/0000-0001-5821-1562,Optimizing the Supply Chain Design for Sourcin...,A significant supply disruption occurred in 20...,AZ; AstraZeneca; B,
1,http://orcid.org/0000-0001-6681-5710,Understanding and Reshaping Social Networks wi...,Social networks are powerful in modeling inter...,NaN,
2,http://orcid.org/0000-0001-8849-4268,Urban Planning and Religious Practice: Three C...,Religion in urban planning is conventionally v...,Western; non-Western; Iraq; Albania; Protestan...,
3,http://orcid.org/0000-0002-0163-9215,Low Power Adaptive Wireless Circuits for the I...,The emergence of the Internet of Things (IoT) ...,Bluetooth Low Energy; Internet of Things; IoT,
4,http://orcid.org/0000-0002-0421-890X,Probabilistic modeling and Bayesian inference ...,Probabilistic modeling and Bayesian inference ...,ov; Hessian; Bayesian; Mark; Gaussian; non,


## Strategy 1: one vector per author, top-k nearest ILC level 2 centroids

For each ORCID, all article `smart_features` are concatenated into one author profile text. The author vector is compared with ILC level 2 centroids, and the top-k closest centroids are saved.

In [26]:
author_profiles = (
    df_with_orcid.groupby("orcid")
    .agg(
        works_count=("orcid", "count"),
        author_smart_features=("smart_features", lambda values: " ".join(values.fillna("").astype(str))),
    )
    .reset_index()
)

if AUTHOR_EMBEDDINGS_PATH.exists():
    author_embeddings = np.load(AUTHOR_EMBEDDINGS_PATH)
    if author_embeddings.shape[0] != len(author_profiles):
        print("Old author embeddings size mismatch, recomputing...")
        author_embeddings = encode_texts(author_profiles["author_smart_features"].fillna("").astype(str).tolist(), model, BATCH_SIZE, "author embeddings")
        np.save(AUTHOR_EMBEDDINGS_PATH, author_embeddings)
else:
    author_embeddings = encode_texts(author_profiles["author_smart_features"].fillna("").astype(str).tolist(), model, BATCH_SIZE, "author embeddings")
    np.save(AUTHOR_EMBEDDINGS_PATH, author_embeddings)

author_embeddings_scaled_for_pca = scaler_pca.transform(author_embeddings)
author_embeddings_reduced = pca.transform(author_embeddings_scaled_for_pca)
author_distances = pairwise_distances(author_embeddings_reduced, ilc_centroids_reduced, metric="cosine")
topk_indices = np.argsort(author_distances, axis=1)[:, :TOP_K_CENTROIDS]

author_topk_rows = []
for author_idx, row in author_profiles.iterrows():
    top_parts = []
    out = {
        "orcid": row["orcid"],
        "works_count": int(row["works_count"]),
    }
    for rank, centroid_idx in enumerate(topk_indices[author_idx], start=1):
        centroid = df_centroids.iloc[int(centroid_idx)]
        distance = float(author_distances[author_idx, centroid_idx])
        out[f"top{rank}_ilc_notation"] = centroid["notation"]
        out[f"top{rank}_ilc_label"] = centroid["label"]
        out[f"top{rank}_distance"] = distance
        top_parts.append(f"{centroid['notation']}:{centroid['label']}:{distance:.4f}")
    out["topk_centroids"] = "; ".join(top_parts)
    author_topk_rows.append(out)

df_author_topk = pd.DataFrame(author_topk_rows)
df_author_topk.to_csv(AUTHOR_TOPK_CSV, index=False)
print("Author top-k saved:", AUTHOR_TOPK_CSV)
df_author_topk.head()

author embeddings: 100%|██████████| 181/181 [00:03<00:00, 59.09it/s]


Author top-k saved: ..\data\artifacts\author_topk_ilc_centroids.csv


,orcid,works_count,top1_ilc_notation,top1_ilc_label,top1_distance,top2_ilc_notation,top2_ilc_label,top2_distance,top3_ilc_notation,top3_ilc_label,top3_distance,top4_ilc_notation,top4_ilc_label,top4_distance,top5_ilc_notation,top5_ilc_label,top5_distance,topk_centroids
0,http://orcid.org/0000-0001-5821-1562,1,kf,tRNA,0.093689,mg,rhizaria,0.097119,rwzs,Scientology,0.119508,aWz,world,0.136194,sf,families,0.140926,kf:tRNA:0.0937; mg:rhizaria:0.0971; rwzs:Scien...
1,http://orcid.org/0000-0001-6681-5710,1,kf,tRNA,0.093689,mg,rhizaria,0.097119,rwzs,Scientology,0.119508,aWz,world,0.136194,sf,families,0.140926,kf:tRNA:0.0937; mg:rhizaria:0.0971; rwzs:Scien...
2,http://orcid.org/0000-0001-8849-4268,1,kf,tRNA,0.093689,mg,rhizaria,0.097119,rwzs,Scientology,0.119508,aWz,world,0.136194,sf,families,0.140926,kf:tRNA:0.0937; mg:rhizaria:0.0971; rwzs:Scien...
3,http://orcid.org/0000-0002-0163-9215,1,kf,tRNA,0.093689,mg,rhizaria,0.097119,rwzs,Scientology,0.119508,aWz,world,0.136194,sf,families,0.140926,kf:tRNA:0.0937; mg:rhizaria:0.0971; rwzs:Scien...
4,http://orcid.org/0000-0002-0421-890X,1,kf,tRNA,0.093689,mg,rhizaria,0.097119,rwzs,Scientology,0.119508,aWz,world,0.136194,sf,families,0.140926,kf:tRNA:0.0937; mg:rhizaria:0.0971; rwzs:Scien...


## Strategy 2: article vectors first, then author summaries

Each author's articles are vectorized and assigned to ILC level 2 centroids. Then the notebook builds author-level summaries: primary centroid, number of distinct centroids, average distance, and top centroids by article count.

In [27]:
if WITH_ORCID_EMBEDDINGS_PATH.exists():
    with_orcid_embeddings = np.load(WITH_ORCID_EMBEDDINGS_PATH)
    if with_orcid_embeddings.shape[0] != len(df_with_orcid):
        print("Old with-orcid embeddings size mismatch, recomputing...")
        with_orcid_embeddings = encode_texts(df_with_orcid["smart_features"].fillna("").astype(str).tolist(), model, BATCH_SIZE, "with-orcid article embeddings")
        np.save(WITH_ORCID_EMBEDDINGS_PATH, with_orcid_embeddings)
else:
    with_orcid_embeddings = encode_texts(df_with_orcid["smart_features"].fillna("").astype(str).tolist(), model, BATCH_SIZE, "with-orcid article embeddings")
    np.save(WITH_ORCID_EMBEDDINGS_PATH, with_orcid_embeddings)

with_orcid_embeddings_scaled_for_pca = scaler_pca.transform(with_orcid_embeddings)
with_orcid_embeddings_reduced = pca.transform(with_orcid_embeddings_scaled_for_pca)
with_orcid_distances = pairwise_distances(with_orcid_embeddings_reduced, ilc_centroids_reduced, metric="cosine")
with_orcid_labels = with_orcid_distances.argmin(axis=1)

df_with_orcid_result = df_with_orcid.copy()
df_with_orcid_result["ilc_level2_cluster"] = with_orcid_labels
df_with_orcid_result["min_distance_to_ilc_center"] = with_orcid_distances.min(axis=1)
df_with_orcid_result["ilc_notation"] = df_with_orcid_result["ilc_level2_cluster"].map(lambda i: df_centroids.iloc[int(i)]["notation"])
df_with_orcid_result["ilc_label"] = df_with_orcid_result["ilc_level2_cluster"].map(lambda i: df_centroids.iloc[int(i)]["label"])
df_with_orcid_result.to_csv(WITH_ORCID_LEVEL2_CSV, index=False)
print("With-ORCID article assignments saved:", WITH_ORCID_LEVEL2_CSV)

summary_rows = []
for orcid, group in df_with_orcid_result.groupby("orcid"):
    counts = (
        group.groupby(["ilc_notation", "ilc_label"])
        .size()
        .reset_index(name="works_count")
        .sort_values("works_count", ascending=False)
    )
    primary = counts.iloc[0]
    top_counts = counts.head(TOP_K_CENTROIDS)
    summary_rows.append({
        "orcid": orcid,
        "works_count": int(len(group)),
        "unique_ilc_centroids": int(len(counts)),
        "primary_ilc_notation": primary["ilc_notation"],
        "primary_ilc_label": primary["ilc_label"],
        "primary_works_count": int(primary["works_count"]),
        "mean_distance_to_ilc_center": float(group["min_distance_to_ilc_center"].mean()),
        "median_distance_to_ilc_center": float(group["min_distance_to_ilc_center"].median()),
        "top_centroids_by_works": "; ".join(
            f"{r.ilc_notation}:{r.ilc_label}:{int(r.works_count)}" for r in top_counts.itertuples(index=False)
        ),
    })

df_author_works_summary = pd.DataFrame(summary_rows).sort_values("works_count", ascending=False)
df_author_works_summary.to_csv(AUTHOR_WORKS_SUMMARY_CSV, index=False)
print("Author works summary saved:", AUTHOR_WORKS_SUMMARY_CSV)
df_author_works_summary.head()

with-orcid article embeddings: 100%|██████████| 181/181 [00:02<00:00, 74.39it/s]


With-ORCID article assignments saved: ..\data\artifacts\with_orcid_level2_assignments.csv
Author works summary saved: ..\data\artifacts\author_works_summary.csv


,orcid,works_count,unique_ilc_centroids,primary_ilc_notation,primary_ilc_label,primary_works_count,mean_distance_to_ilc_center,median_distance_to_ilc_center,top_centroids_by_works
0,http://orcid.org/0000-0001-5821-1562,1,1,kf,tRNA,1,0.093689,0.093689,kf:tRNA:1
1,http://orcid.org/0000-0001-6681-5710,1,1,kf,tRNA,1,0.093689,0.093689,kf:tRNA:1
2,http://orcid.org/0000-0001-8849-4268,1,1,kf,tRNA,1,0.093689,0.093689,kf:tRNA:1
3,http://orcid.org/0000-0002-0163-9215,1,1,kf,tRNA,1,0.093689,0.093689,kf:tRNA:1
4,http://orcid.org/0000-0002-0421-890X,1,1,kf,tRNA,1,0.093689,0.093689,kf:tRNA:1


## Baseline: Standard KMeans without fixed centroids

Run standard KMeans with the same number of clusters (n_clusters = number of ILC level 2 centroids) but without initialization from ILC centroids. This allows comparison of clustering quality with and without domain knowledge.

In [28]:
kmeans_baseline = KMeans(
    n_clusters=len(ilc_centroids_reduced),
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=42,
)

baseline_labels = kmeans_baseline.fit_predict(article_embeddings_reduced)
baseline_distances = pairwise_distances(article_embeddings_reduced, kmeans_baseline.cluster_centers_, metric="cosine")

baseline_quality = {
    "n_articles": int(len(baseline_labels)),
    "n_centroids": int(len(kmeans_baseline.cluster_centers_)),
    "n_used_clusters": int(len(np.unique(baseline_labels))),
    "n_empty_clusters": int(len(kmeans_baseline.cluster_centers_) - len(np.unique(baseline_labels))),
    "mean_assigned_cosine_distance": float(np.mean(baseline_distances.min(axis=1))),
    "median_assigned_cosine_distance": float(np.median(baseline_distances.min(axis=1))),
    "p95_assigned_cosine_distance": float(np.percentile(baseline_distances.min(axis=1), 95)),
    "kmeans_matches_nearest_original_centroid_ratio": float(np.mean(baseline_labels == nearest_original_centroid)),
}

metric_embeddings_baseline = article_embeddings_reduced
metric_labels_baseline = baseline_labels
if len(metric_labels_baseline) > QUALITY_SAMPLE_SIZE:
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(metric_labels_baseline), size=QUALITY_SAMPLE_SIZE, replace=False)
    metric_embeddings_baseline = metric_embeddings_baseline[sample_idx]
    metric_labels_baseline = metric_labels_baseline[sample_idx]

baseline_quality["quality_sample_size"] = int(len(metric_labels_baseline))
metric_unique_labels_baseline = np.unique(metric_labels_baseline)
if 1 < len(metric_unique_labels_baseline) < len(metric_labels_baseline):
    baseline_quality["silhouette_cosine"] = float(silhouette_score(metric_embeddings_baseline, metric_labels_baseline, metric="cosine"))
    baseline_quality["davies_bouldin"] = float(davies_bouldin_score(metric_embeddings_baseline, metric_labels_baseline))
    baseline_quality["calinski_harabasz"] = float(calinski_harabasz_score(metric_embeddings_baseline, metric_labels_baseline))
else:
    baseline_quality["silhouette_cosine"] = np.nan
    baseline_quality["davies_bouldin"] = np.nan
    baseline_quality["calinski_harabasz"] = np.nan

baseline_quality_df = pd.DataFrame([baseline_quality])
BASELINE_CLUSTERING_QUALITY_CSV = ARTIFACTS / "articles_with_baseline_kmeans_quality.csv"
baseline_quality_df.to_csv(BASELINE_CLUSTERING_QUALITY_CSV, index=False)
print("Baseline quality saved:", BASELINE_CLUSTERING_QUALITY_CSV)
baseline_quality_df

Baseline quality saved: ..\data\artifacts\articles_with_baseline_kmeans_quality.csv


,n_articles,n_centroids,n_used_clusters,n_empty_clusters,mean_assigned_cosine_distance,median_assigned_cosine_distance,p95_assigned_cosine_distance,kmeans_matches_nearest_original_centroid_ratio,quality_sample_size,silhouette_cosine,davies_bouldin,calinski_harabasz
0,124282,333,333,0,0.08371,0.072332,0.186811,0.010162,10000,0.120719,1.758095,159.24266


## Comparison: ILC fixed centroids vs Standard KMeans baseline


In [29]:

comparison_df = pd.concat([
    quality_df.assign(method="ILC fixed centroids"),
    baseline_quality_df.assign(method="Standard KMeans"),
], ignore_index=True)

comparison_df = comparison_df[["method"] + [col for col in comparison_df.columns if col != "method"]]
print("Comparison of clustering methods:")
print(comparison_df.T)

COMPARISON_CSV = ARTIFACTS / "ilc_level2_kmeans_comparison.csv"
comparison_df.to_csv(COMPARISON_CSV, index=False)
print("\nComparison saved:", COMPARISON_CSV)
comparison_df


Comparison of clustering methods:
                                                                  0  \
method                                          ILC fixed centroids   
n_articles                                                   124282   
n_centroids                                                     333   
n_used_clusters                                                 333   
n_empty_clusters                                                  0   
mean_assigned_cosine_distance                              0.272159   
median_assigned_cosine_distance                            0.230579   
p95_assigned_cosine_distance                               0.612932   
kmeans_matches_nearest_original_centroid_ratio             0.521789   
quality_sample_size                                           10000   
silhouette_cosine                                          0.052949   
davies_bouldin                                             1.836095   
calinski_harabasz                          

,method,n_articles,n_centroids,n_used_clusters,n_empty_clusters,mean_assigned_cosine_distance,median_assigned_cosine_distance,p95_assigned_cosine_distance,kmeans_matches_nearest_original_centroid_ratio,quality_sample_size,silhouette_cosine,davies_bouldin,calinski_harabasz
0,ILC fixed centroids,124282,333,333,0,0.272159,0.230579,0.612932,0.521789,10000,0.052949,1.836095,115.364325
1,Standard KMeans,124282,333,333,0,0.083710,0.072332,0.186811,0.010162,10000,0.120719,1.758095,159.242660


## Level 1 ILC Analysis: Clustering Level 2 centroids by Level 1 groups

This section analyzes the ILC hierarchy by grouping Level 2 centroids under their Level 1 parents and performing intra-group analysis of clusters.


In [30]:

# Collect Level 1 centroids
level1_nodes = []
for root in ilc_tree:
    level1_nodes.append(root)

def build_level1_text(node):
    parts = [node.get("description", ""), node.get("label", "")]
    return " ".join(str(part).strip() for part in parts if str(part).strip())

df_centroids_level1 = pd.DataFrame([
    {
        "notation": node.get("notation"),
        "label": node.get("label"),
        "combined_text": build_level1_text(node),
    }
    for node in level1_nodes
])

print("ILC level 1 centroids:", len(df_centroids_level1))
df_centroids_level1.head()


ILC level 1 centroids: 65


,notation,label,combined_text
0,~,infinity,infinity
1,*,absolute,absolute
2,A,those,those
3,B,the first ones,the first ones
4,C,the second ones,the second ones


In [31]:

# Group Level 2 centroids by their Level 1 parents
level2_to_level1 = {}
for root in ilc_tree:
    level1_notation = root.get("notation")
    for child in root.get("children", []):
        level2_notation = child.get("notation")
        level2_to_level1[level2_notation] = level1_notation

df_centroids["level1_notation"] = df_centroids["notation"].map(level2_to_level1)
df_centroids["level1_label"] = df_centroids["level1_notation"].map(
    lambda x: df_centroids_level1[df_centroids_level1["notation"] == x]["label"].iloc[0] if x else None
)

print("Level 2 centroids grouped by Level 1:")
df_centroids.groupby("level1_label").size().sort_values(ascending=False)


Level 2 centroids grouped by Level 1:


level1_label
rituals                  28
molecules                26
artworks                 23
artifacts                18
forms                    18
technologies             18
land                     16
celestial bodies         15
continuum bodies         15
rocks                    14
language                 13
organisms                13
consciousness            11
populations              11
knowledge                10
energy                   10
atoms                    10
communities              10
instincts                 9
polities                  9
enterprises               8
genes                     7
spacetime                 5
branes                    4
the following             4
something                 4
bacteria                  3
persons by birth time     1
dtype: int64

In [32]:

# Analyze clusters within each Level 1 group
for level1_notation, level1_group in df_centroids.groupby("level1_notation"):
    level1_label = level1_group["level1_label"].iloc[0]
    print(f"\n=== Level 1: {level1_label} ({level1_notation}) ===")
    print(f"Number of Level 2 centroids: {len(level1_group)}")

    # Get articles assigned to these Level 2 clusters
    level2_notations = level1_group["notation"].tolist()
    group_articles = df_result[df_result["ilc_notation"].isin(level2_notations)]

    print(f"Total articles in this Level 1 group: {len(group_articles)}")

    # Summary by Level 2 clusters within this group
    cluster_summary = group_articles.groupby(["ilc_notation", "ilc_label"]).size().reset_index(name="articles_count")
    cluster_summary = cluster_summary.sort_values("articles_count", ascending=False)
    print("Level 2 clusters within this group:")
    for _, row in cluster_summary.iterrows():
        print(f"  {row['ilc_notation']}: {row['ilc_label']} - {row['articles_count']} articles")



=== Level 1: persons by birth time (U) ===
Number of Level 2 centroids: 1
Total articles in this Level 1 group: 15
Level 2 clusters within this group:
  UX: persons by birth millennium - 15 articles

=== Level 1: something (X) ===
Number of Level 2 centroids: 4
Total articles in this Level 1 group: 903
Level 2 clusters within this group:
  XB: x1 - 527 articles
  XC: x2 - 135 articles
  XA: x0 - 134 articles
  XX: what - 107 articles

=== Level 1: the following (Z) ===
Number of Level 2 centroids: 4
Total articles in this Level 1 group: 449
Level 2 clusters within this group:
  ZA: the aforementioned - 195 articles
  ZD: the third next - 142 articles
  ZC: the second next - 73 articles
  ZB: the next following - 39 articles

=== Level 1: forms (a) ===
Number of Level 2 centroids: 18
Total articles in this Level 1 group: 7123
Level 2 clusters within this group:
  at: algorithms - 2122 articles
  am: combinatorial structures - 823 articles
  aw: systems - 805 articles
  al: algebraic st

In [33]:

# Function to compute log-odds ratio for words in a cluster vs overall corpus
def compute_log_odds_ratio(cluster_texts, corpus_texts, top_n=50):
    """
    Compute log-odds ratio for words in cluster vs overall corpus.
    Returns top_n words with highest log-odds ratio.
    """
    from collections import Counter
    import re
    import numpy as np
    # Tokenize and count words in cluster
    cluster_words = []
    for text in cluster_texts:
        words = re.findall(r'\b\w+\b', text.lower())
        cluster_words.extend(words)

    cluster_counts = Counter(cluster_words)
    cluster_total = len(cluster_words)

    # Tokenize and count words in corpus
    corpus_words = []
    for text in corpus_texts:
        words = re.findall(r'\b\w+\b', text.lower())
        corpus_words.extend(words)

    corpus_counts = Counter(corpus_words)
    corpus_total = len(corpus_words)

    # Compute log-odds ratio
    log_odds = {}
    for word in set(cluster_words + corpus_words):
        if word not in cluster_counts:
            cluster_counts[word] = 0
        if word not in corpus_counts:
            corpus_counts[word] = 0

        # Add smoothing
        cluster_freq = (cluster_counts[word] + 1) / (cluster_total + len(corpus_counts))
        corpus_freq = (corpus_counts[word] + 1) / (corpus_total + len(corpus_counts))

        odds_ratio = cluster_freq / corpus_freq
        log_odds[word] = np.log(odds_ratio)

    # Return top_n words with highest log-odds
    sorted_words = sorted(log_odds.items(), key=lambda x: x[1], reverse=True)
    return dict(sorted_words[:top_n])

# Create results_odds directory
RESULTS_ODDS_DIR = ARTIFACTS / "results_odds"
RESULTS_ODDS_DIR.mkdir(parents=True, exist_ok=True)

# Function to generate word cloud from log-odds
def generate_wordcloud(log_odds_dict, title="Word Cloud", filename=None):
    from wordcloud import WordCloud
    import matplotlib.pyplot as plt
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(log_odds_dict)

    # Save to file only, do not display
    if filename:
        # Use matplotlib to save without displaying
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(wordcloud, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(title)
        plt.savefig(filename, bbox_inches='tight', dpi=150)
        plt.close(fig)
        print(f"Saved word cloud to: {filename}")
    else:

        print(f"No filename provided for: {title}")

    

In [34]:
import re

In [35]:

# Compute log-odds ratio and generate word clouds for each Level 2 cluster
corpus_texts = df_articles["smart_features"].fillna("").tolist()

for level1_notation, level1_group in df_centroids.groupby("level1_notation"):
    level1_label = level1_group["level1_label"].iloc[0]
    print(f"\n=== Generating word clouds for Level 1: {level1_label} ===")

    for _, level2_row in level1_group.iterrows():
        level2_notation = level2_row["notation"]
        level2_label = level2_row["label"]

        # Get articles in this Level 2 cluster
        cluster_articles = df_result[df_result["ilc_notation"] == level2_notation]
        cluster_texts = cluster_articles["smart_features"].fillna("").tolist()

        if len(cluster_texts) < 10:  # Skip clusters with too few articles
            print(f"Skipping {level2_notation}: {level2_label} - only {len(cluster_texts)} articles")
            continue

        print(f"Processing {level2_notation}: {level2_label} - {len(cluster_texts)} articles")

        # Compute log-odds ratio
        log_odds = compute_log_odds_ratio(cluster_texts, corpus_texts, top_n=50)

        # Generate word cloud and save to file
        title = f"Level 2: {level2_label} ({level2_notation})\n{len(cluster_texts)} articles"
        filename = RESULTS_ODDS_DIR / f"wordcloud_{level1_notation}_{level2_notation}.png"
        generate_wordcloud(log_odds, title=title, filename=str(filename))


=== Generating word clouds for Level 1: persons by birth time ===
Processing UX: persons by birth millennium - 15 articles
Saved word cloud to: ..\data\artifacts\results_odds\wordcloud_U_UX.png

=== Generating word clouds for Level 1: something ===
Processing XA: x0 - 134 articles
Saved word cloud to: ..\data\artifacts\results_odds\wordcloud_X_XA.png
Processing XB: x1 - 527 articles
Saved word cloud to: ..\data\artifacts\results_odds\wordcloud_X_XB.png
Processing XC: x2 - 135 articles
Saved word cloud to: ..\data\artifacts\results_odds\wordcloud_X_XC.png
Processing XX: what - 107 articles
Saved word cloud to: ..\data\artifacts\results_odds\wordcloud_X_XX.png

=== Generating word clouds for Level 1: the following ===
Processing ZA: the aforementioned - 195 articles
Saved word cloud to: ..\data\artifacts\results_odds\wordcloud_Z_ZA.png
Processing ZB: the next following - 39 articles
Saved word cloud to: ..\data\artifacts\results_odds\wordcloud_Z_ZB.png
Processing ZC: the second next - 7